In [1]:
from pathlib import Path
import re
import shutil
import pandas as pd

# ====== Konfigurasi ======
SPLIT_DIR = Path(r"C:\Users\s2\Documents\FIXRKI\FIXbanget\dataset\split")
CLIENT_DIR = SPLIT_DIR / "clientsMalacious"

NEW_CLIENT_DIR = Path(
    r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V3"
) / "clientsMalacious_Selected"

TOP21_STATES = [
    53, 24, 39, 12, 36, 27,
    48, 18, 45, 31, 4
]
SPLITS = ("train", "val")
REQUIRED_FILES = tuple(f"{s}.csv" for s in SPLITS)

SUFFIX = "sinatetic"   # hanya dipakai untuk membersihkan folder lama

COPY_FILES = True
OVERWRITE = False
CLEAN_OLD_SINATETIC = True   # hapus folder state_10xx_sinatetic dari run sebelumnya

if not CLIENT_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {CLIENT_DIR}")

# ====== 1. Deteksi semua state ======
all_states = sorted(
    int(m.group(1))
    for p in CLIENT_DIR.iterdir()
    if p.is_dir() and (m := re.fullmatch(r"state_(\d+)", p.name))
)

NEW_STATES = [s for s in all_states if s not in TOP21_STATES]

print(f"State di disk : {len(all_states)}")
print(f"TOP21         : {len(TOP21_STATES)}")
print(f"State sisa    : {len(NEW_STATES)} -> {NEW_STATES} (tidak disalin)")

# ====== 2. Validasi 21 client terpilih ======
selected_clients = {
    s: CLIENT_DIR / f"state_{s:02d}"
    for s in TOP21_STATES
}

missing = [
    str(path / f)
    for path in selected_clients.values()
    for f in REQUIRED_FILES
    if not (path / f).is_file()
]

if missing:
    raise FileNotFoundError(
        "File berikut tidak ditemukan:\n" + "\n".join(missing)
    )

# ====== 3. Muat data 21 client ======
clients_data = {
    s: {
        split: pd.read_csv(path / f"{split}.csv")
        for split in SPLITS
    }
    for s, path in selected_clients.items()
}

# ====== 4. Ringkasan ======
print(f"\nBerhasil memuat {len(clients_data)} client.\n")

print(f"{'State':<8}{'Train':>10}{'Val':>10}{'Kolom':>8}")

for s, d in clients_data.items():
    print(
        f"{s:02d}".ljust(8)
        + f"{len(d['train']):>10}"
        + f"{len(d['val']):>10}"
        + f"{d['train'].shape[1]:>8}"
    )

# ====== 5. Cek konsistensi kolom ======
ref_cols = list(clients_data[TOP21_STATES[0]]["train"].columns)

for s, d in clients_data.items():
    for split, df in d.items():
        if list(df.columns) != ref_cols:
            print(
                f"⚠️ Kolom state_{s:02d}/{split} "
                "berbeda dari client pertama"
            )

# ====== 6. Salin 21 client TOP21 ke folder Selected ======
NEW_CLIENT_DIR.mkdir(parents=True, exist_ok=True)

# Bersihkan sisa folder _sinatetic dari run sebelumnya
if CLEAN_OLD_SINATETIC:
    old_dirs = list(NEW_CLIENT_DIR.glob(f"state_*_{SUFFIX}"))
    for p in old_dirs:
        shutil.rmtree(p)
    print(f"\nFolder lama dihapus: {len(old_dirs)}")

copied = skipped = absent = 0

for s in TOP21_STATES:
    src_dir = CLIENT_DIR / f"state_{s:02d}"
    dst_dir = NEW_CLIENT_DIR / f"state_{s:02d}"
    dst_dir.mkdir(parents=True, exist_ok=True)

    print(f"state_{s:02d} -> {dst_dir.name}")

    if not COPY_FILES:
        continue

    for f in REQUIRED_FILES:
        src = src_dir / f
        dst = dst_dir / f

        if not src.is_file():
            absent += 1
            print(f"⚠️ Tidak ada di sumber: state_{s:02d}/{f}")
        elif dst.exists() and not OVERWRITE:
            skipped += 1
        else:
            shutil.copy2(src, dst)
            copied += 1

# ====== 7. Verifikasi hasil ======
n_dirs = sum(
    1 for s in TOP21_STATES
    if (NEW_CLIENT_DIR / f"state_{s:02d}").is_dir()
)
n_csv = sum(
    1
    for s in TOP21_STATES
    for f in REQUIRED_FILES
    if (NEW_CLIENT_DIR / f"state_{s:02d}" / f).is_file()
)

# Cek tidak ada folder asing di folder Selected
expected = {f"state_{s:02d}" for s in TOP21_STATES}
extra = sorted(
    p.name for p in NEW_CLIENT_DIR.iterdir()
    if p.is_dir() and p.name not in expected
)

print("\n========== HASIL ==========")
print(f"Folder client baru : {NEW_CLIENT_DIR}")
print(f"Folder state       : {n_dirs} (target {len(TOP21_STATES)})")
print(f"CSV disalin        : {copied}")
print(f"CSV dilewati       : {skipped}")
print(f"CSV tidak tersedia : {absent}")
print(f"Total train/val CSV: {n_csv} (target {len(TOP21_STATES) * len(SPLITS)})")
if extra:
    print(f"⚠️ Folder tak terduga di Selected: {extra}")

State di disk : 44
TOP21         : 11
State sisa    : 33 -> [1, 2, 5, 8, 10, 11, 13, 15, 16, 17, 19, 20, 22, 23, 25, 26, 29, 30, 33, 34, 35, 37, 38, 40, 41, 44, 46, 49, 50, 51, 54, 55, 56] (tidak disalin)

Berhasil memuat 11 client.

State        Train       Val   Kolom
53            3531      1850      72
24            2845      1310      72
39            2757      1057      72
12            2330      1257      72
36            2212      3166      72
27            2163      1047      72
48            1987       998      72
18            1906      1350      72
45            1812      1044      72
31            1790      1310      72
04            1781       804      72

Folder lama dihapus: 0
state_53 -> state_53
state_24 -> state_24
state_39 -> state_39
state_12 -> state_12
state_36 -> state_36
state_27 -> state_27
state_48 -> state_48
state_18 -> state_18
state_45 -> state_45
state_31 -> state_31
state_04 -> state_04

========== HASIL ==========
Folder client baru : C:\Users\s2\Docum

In [2]:
from LabelReverse import reverse_client_labels, compare_outputs

CLIENTS_DIR = r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V3\clientsMalacious_Selected"
NEW_DIR     = r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V3\clientsMalacious_Reverse"

reverse_client_labels(CLIENTS_DIR, label_col="_michd", output_dir=NEW_DIR)
compare_outputs(CLIENTS_DIR, NEW_DIR, label_col="_michd")

state_04
  train.csv: {0: 1390, 1: 391} -> {0: 391, 1: 1390}
  val.csv: {0: 634, 1: 170} -> {0: 170, 1: 634}
state_12
  train.csv: {0: 1741, 1: 589} -> {0: 589, 1: 1741}
  val.csv: {0: 983, 1: 274} -> {0: 274, 1: 983}
state_18
  train.csv: {0: 1513, 1: 393} -> {0: 393, 1: 1513}
  val.csv: {0: 1036, 1: 314} -> {0: 314, 1: 1036}
state_24
  train.csv: {0: 2292, 1: 553} -> {0: 553, 1: 2292}
  val.csv: {0: 1053, 1: 257} -> {0: 257, 1: 1053}
state_27
  train.csv: {0: 1739, 1: 424} -> {0: 424, 1: 1739}
  val.csv: {0: 811, 1: 236} -> {0: 236, 1: 811}
state_31
  train.csv: {0: 1421, 1: 369} -> {0: 369, 1: 1421}
  val.csv: {0: 1012, 1: 298} -> {0: 298, 1: 1012}
state_36
  train.csv: {0: 1766, 1: 446} -> {0: 446, 1: 1766}
  val.csv: {0: 2493, 1: 673} -> {0: 673, 1: 2493}
state_39
  train.csv: {0: 2091, 1: 666} -> {0: 666, 1: 2091}
  val.csv: {0: 804, 1: 253} -> {0: 253, 1: 804}
state_45
  train.csv: {0: 1443, 1: 369} -> {0: 369, 1: 1443}
  val.csv: {0: 807, 1: 237} -> {0: 237, 1: 807}
state_48
  

,state,split,asli_0,asli_1,baru_0,baru_1,semua_baris_terbalik
0,04,train,1390,391,391,1390,True
1,04,val,634,170,170,634,True
2,12,train,1741,589,589,1741,True
3,12,val,983,274,274,983,True
4,18,train,1513,393,393,1513,True
5,18,val,1036,314,314,1036,True
6,24,train,2292,553,553,2292,True
7,24,val,1053,257,257,1053,True
8,27,train,1739,424,424,1739,True
9,27,val,811,236,236,811,True


In [3]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\s2\Documents\FIXRKI\FIXbanget\ScenarioMalacious\ML2V3")
STATE = 53          # harus salah satu dari TOP21_STATES
SUFFIX = ""        # "" -> state_04 | "sinatetic" -> state_04_sinatetic
LABEL = "_michd"

folder = f"state_{STATE:02d}" + (f"_{SUFFIX}" if SUFFIX else "")

path_awal = BASE / "clientsMalacious_Selected" / folder / "train.csv"
path_baru = BASE / "clientsMalacious_Reverse" / folder / "train.csv"

for p in (path_awal, path_baru):
    if not p.is_file():
        raise FileNotFoundError(f"File tidak ditemukan: {p}")

data_awal = pd.read_csv(path_awal)
data_baru = pd.read_csv(path_baru)

print("Data Awal:\n", data_awal[LABEL].head())
print("\nData Baru:\n", data_baru[LABEL].head())

# Verifikasi: semua baris harus terbalik
print("\nJumlah baris sama :", len(data_awal) == len(data_baru))
print("Semua terbalik    :", bool(((data_awal[LABEL] + data_baru[LABEL]) == 1).all()))

Data Awal:
 0    0
1    0
2    0
3    0
4    0
Name: _michd, dtype: int64

Data Baru:
 0    1
1    1
2    1
3    1
4    1
Name: _michd, dtype: int64

Jumlah baris sama : True
Semua terbalik    : True


In [4]:
print("Data Awal:", data_awal["_michd"].value_counts())
print("Data Baru:", data_baru["_michd"].value_counts())

Data Awal: _michd
0    2830
1     701
Name: count, dtype: int64
Data Baru: _michd
1    2830
0     701
Name: count, dtype: int64
